# Modelo genérico · Árvore de Decisão implementada do zero

**Template reutilizável** de classificação com árvore de decisão: impureza (entropia ou Gini), divisão de um nó, ganho de informação, melhor divisão e construção recursiva, com **previsão**, regras legíveis, avaliação e comparação com o `DecisionTreeClassifier` do scikit-learn.

| | |
|---|---|
| **Modelo no sistema** | Machine Learning › 2. Aprendizado supervisionado › cartão *Decision Tree* (`/modelar/aprendizado-supervisionado/#m-decision-tree`) |
| **Origem** | `C2_W4_Decision_Tree_with_Markdown.ipynb` (*Machine Learning Specialization*, DeepLearning.AI e Stanford Online; cogumelos comestíveis × venenosos, 10 exemplos e 3 features binárias), revisado, corrigido e generalizado |
| **Tipo de problema** | Supervisionado · classificação (binária ou multiclasse) |
| **Dependências** | `numpy`, `pandas`, `matplotlib`, `scikit-learn` |

### Como usar
1. Edite apenas a célula **1. Configuração** (`>>> ADAPTAR`).
2. Com `CAMINHO_DADOS = None`, roda com dados sintéticos (features numéricas e uma categórica).
3. Features categóricas viram colunas 0/1 (one-hot); numéricas são divididas por **limiares**. Árvores não precisam de padronização.

### O que mudou em relação ao notebook original
- **Erro corrigido:** quando nenhuma feature tinha ganho positivo (nó puro ou indivisível antes da profundidade máxima), `get_best_split` devolvia `-1`, e `split_dataset` dividia silenciosamente pela **última coluna** (`X[i][-1]`). Agora o nó vira folha.
- O original só aceitava features **binárias 0/1** e alvo binário. Agora aceita features numéricas (limiares candidatos entre valores consecutivos), alvo multiclasse e os critérios entropia ou Gini, que o cartão cita.
- O original só **imprimia** a árvore (numa lista global `tree`) e não tinha previsão. Agora a árvore é uma estrutura de nós, com `predict`, regras legíveis e avaliação em treino e teste.
- Critérios de parada além da profundidade: mínimo de amostras para dividir, nó puro e ganho mínimo.
- `utils.py` (visualizações com as fotos dos cogumelos) e `public_tests.py` foram removidos; os ganhos de informação do exemplo original viraram uma verificação automática.

## 1. Configuração

In [ ]:
# >>> ADAPTAR: única célula que precisa ser editada para usar outro dataset.

# --- dados ---
CAMINHO_DADOS = None        # ex.: "dados/pedidos.csv"; None = dados sintéticos
SEPARADOR = None
TEM_CABECALHO = True        # False se a primeira linha já é dado
NOMES_COLUNAS = None        # nomes quando TEM_CABECALHO = False; sem nomes, viram coluna_0, coluna_1, …
COLUNA_ALVO = "alvo"        # coluna com as classes (2 ou mais)
COLUNAS_FEATURES = None     # None = todas as demais

# --- divisão e reprodutibilidade ---
TAMANHO_TESTE = 0.25
SEMENTE = 42

# --- árvore ---
CRITERIO = "entropia"       # "entropia" (como no original) ou "gini"
PROFUNDIDADE_MAXIMA = 4     # o original parava na profundidade 2
MIN_AMOSTRAS_DIVISAO = 10   # nós com menos amostras viram folha
MIN_GANHO = 1e-7            # ganho de informação mínimo para dividir
MAX_LIMIARES = 32           # limiares candidatos por feature numérica (quantis); controla o custo

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text

## 2. Carregamento dos dados

In [ ]:
def carregar_tabela(caminho, separador=None, tem_cabecalho=True, nomes_colunas=None, planilha=0):
    """Lê CSV/TXT/TSV, Excel, Parquet ou JSON e devolve um DataFrame."""
    caminho = Path(caminho)
    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho.resolve()}")
    ext = caminho.suffix.lower()
    cabecalho = 0 if tem_cabecalho else None
    if ext in {".csv", ".txt", ".tsv"}:
        sep = separador if separador is not None else ("\t" if ext == ".tsv" else None)
        df = pd.read_csv(caminho, sep=sep, engine="python", header=cabecalho)
    elif ext in {".xlsx", ".xls"}:
        df = pd.read_excel(caminho, sheet_name=planilha, header=cabecalho)
    elif ext == ".parquet":
        df = pd.read_parquet(caminho)
    elif ext == ".json":
        df = pd.read_json(caminho)
    else:
        raise ValueError(f"Formato não suportado: {ext}")
    if nomes_colunas is not None:
        df.columns = nomes_colunas
    elif not tem_cabecalho:
        df.columns = [f"coluna_{i}" for i in range(df.shape[1])]
    return df


def dados_sinteticos(n=600, semente=SEMENTE):
    rng = np.random.default_rng(semente)
    df = pd.DataFrame({"feature_1": rng.normal(50, 10, n), "feature_2": rng.uniform(0, 1, n),
                       "feature_3": rng.integers(0, 2, n), "grupo": rng.choice(["A", "B", "C"], n)})
    pontos = (df.feature_1 > 55) * 1.0 + (df.feature_2 > 0.7) * 1.0 + (df.grupo == "C") * 1.2 + rng.normal(0, 0.4, n)
    df["alvo"] = np.select([pontos > 1.6, pontos > 0.6], ["alto", "medio"], "baixo")
    return df


df = (dados_sinteticos() if CAMINHO_DADOS is None
      else carregar_tabela(CAMINHO_DADOS, SEPARADOR, TEM_CABECALHO, NOMES_COLUNAS))
if COLUNA_ALVO not in df.columns:
    raise KeyError(f"COLUNA_ALVO={COLUNA_ALVO!r} não existe. Colunas: {list(df.columns)}")
df = df.dropna(subset=[COLUNA_ALVO])
codificador = LabelEncoder()
y = codificador.fit_transform(df[COLUNA_ALVO])
classes = codificador.classes_
features = COLUNAS_FEATURES or [c for c in df.columns if c != COLUNA_ALVO]
X_df = df[features]
print("formato:", df.shape, "| classes:", list(classes))
display(df.head())

## 3. Preparação

In [ ]:
def montar_preprocessador(X, padronizar=True, remover_primeira=False):
    """Numéricas: mediana + padronização. Categóricas: moda + one-hot.
    remover_primeira=True descarta uma categoria por variável (evita a colinearidade perfeita
    com o intercepto em modelos lineares). Ajuste SÓ no treino (evita vazamento do teste)."""
    numericas = X.select_dtypes(include="number").columns.tolist()
    categoricas = [c for c in X.columns if c not in numericas]
    passos_num = [SimpleImputer(strategy="median")] + ([StandardScaler()] if padronizar else [])
    transformadores = []
    if numericas:
        transformadores.append(("num", make_pipeline(*passos_num), numericas))
    if categoricas:
        transformadores.append(("cat", make_pipeline(
            SimpleImputer(strategy="most_frequent"),
            OneHotEncoder(handle_unknown="ignore", sparse_output=False,
                          drop="first" if remover_primeira else None)), categoricas))
    return ColumnTransformer(transformadores, verbose_feature_names_out=False)


X_tr_df, X_te_df, y_train, y_test = train_test_split(X_df, y, test_size=TAMANHO_TESTE, random_state=SEMENTE, stratify=y)
prep = montar_preprocessador(X_tr_df, padronizar=False)       # árvores não precisam de escala
X_train = prep.fit_transform(X_tr_df)
X_test = prep.transform(X_te_df)
nomes = list(prep.get_feature_names_out())
print("treino:", X_train.shape, "| teste:", X_test.shape)
print("features após one-hot:", nomes)

## 4. Impureza de um nó
Com $p_k$ a fração de exemplos da classe $k$ no nó:

$$H = -\sum_k p_k\log_2 p_k \qquad\qquad \text{Gini} = 1 - \sum_k p_k^2$$

Por convenção $0\log_2 0 = 0$, e um nó vazio tem impureza 0. Com duas classes, a entropia é exatamente a do original: $H(p_1) = -p_1\log_2 p_1 - (1-p_1)\log_2(1-p_1)$.

In [ ]:
def compute_entropy(y):
    """Entropia (base 2) dos rótulos de um nó; 0 se o nó estiver vazio ou puro."""
    if len(y) == 0:
        return 0.0
    p = np.bincount(y) / len(y)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


def compute_gini(y):
    if len(y) == 0:
        return 0.0
    p = np.bincount(y) / len(y)
    return float(1 - (p ** 2).sum())


impureza = compute_entropy if CRITERIO == "entropia" else compute_gini

## 5. Divisão, ganho de informação e melhor divisão
`split_dataset` manda para a **esquerda** os exemplos com `x[feature] >= limiar`; com uma feature 0/1 e limiar 0,5, a esquerda é o valor 1, como no original.

$$\text{Ganho} = I(\text{nó}) - \left(w^{\text{esq}}\,I(\text{esq}) + w^{\text{dir}}\,I(\text{dir})\right)$$

com $w$ a proporção de exemplos de cada ramo. Para features numéricas, os limiares candidatos ficam entre valores consecutivos (no máximo `MAX_LIMIARES`, por quantis).

In [ ]:
def split_dataset(X, node_indices, feature, threshold=0.5):
    """Divide os índices do nó: esquerda = X[i, feature] >= threshold; direita = o resto."""
    node_indices = np.asarray(node_indices)
    vai_esq = X[node_indices, feature] >= threshold
    return node_indices[vai_esq].tolist(), node_indices[~vai_esq].tolist()


def compute_information_gain(X, y, node_indices, feature, threshold=0.5):
    left, right = split_dataset(X, node_indices, feature, threshold)
    if not left or not right:                       # divisão que não separa nada: ganho zero
        return 0.0
    n = len(node_indices)
    ponderada = len(left) / n * impureza(y[left]) + len(right) / n * impureza(y[right])
    return impureza(y[node_indices]) - ponderada


def limiares_candidatos(valores):
    unicos = np.unique(valores)
    if len(unicos) < 2:
        return np.array([])
    meios = (unicos[:-1] + unicos[1:]) / 2
    if len(meios) > MAX_LIMIARES:
        meios = np.unique(np.quantile(meios, np.linspace(0, 1, MAX_LIMIARES)))
    return meios


def get_best_split(X, y, node_indices):
    """Devolve (melhor_feature, melhor_limiar, ganho). melhor_feature = -1 quando nada melhora o nó."""
    melhor = (-1, None, 0.0)
    for feature in range(X.shape[1]):
        for limiar in limiares_candidatos(X[node_indices, feature]):
            ganho = compute_information_gain(X, y, node_indices, feature, limiar)
            if ganho > melhor[2] + MIN_GANHO:
                melhor = (feature, float(limiar), ganho)
    return melhor

### 5.1 Verificação com o exemplo do original
Os 10 exemplos do original (3 features 0/1) e os ganhos de informação que ele documentava na raiz. Se a implementação mudar o resultado, a célula falha.

In [ ]:
X_ex = np.array([[1,1,1],[1,0,1],[1,0,0],[1,0,0],[1,1,1],[0,1,1],[0,0,0],[1,0,1],[0,1,0],[1,0,0]])
y_ex = np.array([1,1,0,0,1,0,0,1,1,0])
raiz = list(range(10))
assert compute_entropy(y_ex) == 1.0
assert split_dataset(X_ex, raiz, 0) == ([0, 1, 2, 3, 4, 7, 9], [5, 6, 8])
ganhos = [compute_entropy(y_ex) - sum(len(r) / 10 * compute_entropy(y_ex[r]) for r in split_dataset(X_ex, raiz, f))
          for f in range(3)]
assert np.allclose(ganhos, [0.034851554559677034, 0.12451124978365313, 0.2780719051126377])
if CRITERIO == "entropia":
    assert np.allclose([compute_information_gain(X_ex, y_ex, raiz, f) for f in range(3)], ganhos)
    assert get_best_split(X_ex, y_ex, raiz)[0] == 2          # "Solitary", como no original
print("entropia, divisão e ganhos conferem com o original:", np.round(ganhos, 4))

## 6. Construção recursiva
Começa com todos os exemplos na raiz, escolhe a divisão de maior ganho, separa em esquerda e direita e repete até um critério de parada. Cada folha guarda a classe majoritária e a distribuição de classes.

In [ ]:
def folha(y, node_indices):
    contagem = np.bincount(y[node_indices], minlength=len(classes))
    return {"folha": True, "classe": int(np.argmax(contagem)), "n": len(node_indices), "dist": contagem}


def build_tree_recursive(X, y, node_indices, branch_name="Raiz", max_depth=PROFUNDIDADE_MAXIMA,
                         current_depth=0, imprimir=True):
    """Constrói a árvore e devolve o nó raiz (dicionário). Imprime a estrutura como o original."""
    recuo = "  " * current_depth + "-" * current_depth
    parar = (current_depth >= max_depth or len(node_indices) < MIN_AMOSTRAS_DIVISAO
             or impureza(y[node_indices]) == 0)
    feature, limiar, ganho = (-1, None, 0.0) if parar else get_best_split(X, y, node_indices)
    if feature == -1:                               # correção: sem divisão útil, o nó vira folha
        no = folha(y, node_indices)
        if imprimir:
            print(f"{recuo} {branch_name}: folha → {classes[no['classe']]} (n = {no['n']})")
        return no
    if imprimir:
        print(f"{recuo} Profundidade {current_depth}, {branch_name}: divide em {nomes[feature]} >= {limiar:.4g} "
              f"(ganho {ganho:.4f})")
    esq, dir_ = split_dataset(X, node_indices, feature, limiar)
    return {"folha": False, "feature": feature, "limiar": limiar, "ganho": ganho, "n": len(node_indices),
            "esq": build_tree_recursive(X, y, esq, "Esquerda", max_depth, current_depth + 1, imprimir),
            "dir": build_tree_recursive(X, y, dir_, "Direita", max_depth, current_depth + 1, imprimir)}


def predict_um(x, no):
    while not no["folha"]:
        no = no["esq"] if x[no["feature"]] >= no["limiar"] else no["dir"]
    return no["classe"]


def predict(X, arvore):
    return np.array([predict_um(x, arvore) for x in X])


def regras(no, prefixo=""):
    """Regras SE … ENTÃO legíveis, uma por folha."""
    if no["folha"]:
        return [f"SE {prefixo or 'sempre'} ENTÃO {classes[no['classe']]} (n = {no['n']})"]
    cond = f"{nomes[no['feature']]} >= {no['limiar']:.4g}"
    neg = f"{nomes[no['feature']]} < {no['limiar']:.4g}"
    juntar = lambda c: f"{prefixo} E {c}" if prefixo else c
    return regras(no["esq"], juntar(cond)) + regras(no["dir"], juntar(neg))

In [ ]:
arvore = build_tree_recursive(X_train, y_train, list(range(len(y_train))))
print()
print(*regras(arvore), sep="\n")

## 7. Avaliação

In [ ]:
pred_tr, pred_te = predict(X_train, arvore), predict(X_test, arvore)
display(pd.DataFrame({"treino": {"acurácia": accuracy_score(y_train, pred_tr), "F1 macro": f1_score(y_train, pred_tr, average="macro")},
                      "teste": {"acurácia": accuracy_score(y_test, pred_te), "F1 macro": f1_score(y_test, pred_te, average="macro")}}).T.round(4))
print(classification_report(y_test, pred_te, target_names=[str(c) for c in classes], digits=3, zero_division=0))
ConfusionMatrixDisplay.from_predictions(classes[y_test], classes[pred_te], colorbar=False)
plt.title("Matriz de confusão (teste)"); plt.show()

### 7.1 Profundidade × sobreajuste
O cartão alerta para a **alta variância** das árvores. A curva compara treino e teste por profundidade; na prática, escolha a profundidade com validação cruzada, não com o teste.

In [ ]:
profundidades = range(1, max(PROFUNDIDADE_MAXIMA, 6) + 3)
curva = []
for d in profundidades:
    a = build_tree_recursive(X_train, y_train, list(range(len(y_train))), max_depth=d, imprimir=False)
    curva.append((d, accuracy_score(y_train, predict(X_train, a)), accuracy_score(y_test, predict(X_test, a))))
curva = pd.DataFrame(curva, columns=["profundidade", "treino", "teste"]).set_index("profundidade")
curva.plot(marker="o", figsize=(6, 3.5), title="Acurácia por profundidade máxima"); plt.ylabel("acurácia"); plt.show()

## 8. Validação com a referência (scikit-learn)
Mesmo critério e mesmos limites. Pequenas diferenças vêm de empates entre divisões e dos limiares por quantis (`MAX_LIMIARES`); a acurácia e a maior parte das previsões devem coincidir.

In [ ]:
ref = DecisionTreeClassifier(criterion="entropy" if CRITERIO == "entropia" else "gini", max_depth=PROFUNDIDADE_MAXIMA,
                             min_samples_split=MIN_AMOSTRAS_DIVISAO, random_state=SEMENTE).fit(X_train, y_train)
pred_ref = ref.predict(X_test)
print(f"acurácia teste · do zero: {accuracy_score(y_test, pred_te):.4f} | scikit-learn: {accuracy_score(y_test, pred_ref):.4f}")
print(f"concordância das previsões: {np.mean(pred_ref == pred_te):.1%}")
print(export_text(ref, feature_names=nomes, class_names=[str(c) for c in classes]))

## 9. Previsão para novos dados

In [ ]:
novos = X_df.sample(3, random_state=SEMENTE)     # >>> ADAPTAR: DataFrame com as mesmas colunas de `features`
display(novos.assign(previsao=classes[predict(prep.transform(novos), arvore)]))

## 10. Próximos passos (segundo o cartão do sistema)
- Hiperparâmetros: `max_depth`, `min_samples_split`, `min_samples_leaf`, `max_features`, `ccp_alpha` (poda), `criterion`, `class_weight`.
- Para **regressão**, a impureza vira a variância do alvo no nó (`DecisionTreeRegressor`).
- **Prefira Random Forest ou boosting** sempre que o desempenho importar mais que a leitura literal das regras.